# Telco Customer Churn — Analytical Walkthrough

This notebook is the **narrative companion** to the scripted pipeline in `src/`. It walks through the same analysis interactively: data → cleaning → EDA → modeling → business impact.

> Run order: top to bottom. All heavy lifting lives in the `src/` modules so this notebook stays readable.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
import config
from data_preprocessing import build_processed_dataset

pd.set_option('display.max_columns', None)
config.set_theme()

## 1. Load & clean
We fix the text-typed `TotalCharges`, normalise categories, drop the ID, and engineer lifecycle/engagement features.

In [2]:
df = build_processed_dataset(save=True)
df.head()

[load]   raw shape: 7,043 rows x 21 cols
[clean]  fixed 11 blank TotalCharges (new, tenure=0 customers)
[clean]  duplicates removed: 22
[feateng] engineered features -> new shape: 26 cols


[save]   processed dataset -> C:\Users\HP VICTUS\Desktop\project 2\data\processed\telco_churn_clean.csv
[done]   final shape: 7,021 x 26 | churn rate: 26.4%


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,NumAddOnServices,AvgChargesPerMonth,IsMonthToMonth,IsElectronicCheck,HasFiberOptic
0,Female,No,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-1 yr,1,29.850000,1,1,0
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,2-4 yr,2,55.573529,0,0,0
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-1 yr,2,54.075000,1,0,0
3,Male,No,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,2-4 yr,3,40.905556,0,0,0
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-1 yr,0,75.825000,1,1,1


In [3]:
print('Customers:', len(df))
print('Churn rate: {:.1%}'.format((df.Churn == 'Yes').mean()))
df[['tenure','MonthlyCharges','TotalCharges','NumAddOnServices']].describe().round(2)

Customers: 7021
Churn rate: 26.4%


,tenure,MonthlyCharges,TotalCharges,NumAddOnServices
count,7021.00,7021.00,7021.00,7021.00
mean,32.47,64.85,2286.77,2.04
std,24.53,30.07,2266.86,1.85
min,0.00,18.25,0.00,0.00
25%,9.00,35.75,403.35,0.00
50%,29.00,70.40,1400.55,2.00
75%,55.00,89.90,3801.70,3.00
max,72.00,118.75,8684.80,6.00


## 2. Exploratory data analysis
Regenerate the full polished figure set into `reports/figures/`.

In [4]:
from eda import run_all_eda
run_all_eda()

[eda] generating figures...


[fig]  01_churn_overview.png


[fig]  02_churn_drivers.png


[fig]  03_tenure_and_charges.png


[fig]  04_correlation_heatmap.png


[fig]  05_contract_payment_matrix.png
[eda] done -> C:\Users\HP VICTUS\Desktop\project 2\reports\figures


In [5]:
# Churn rate by the strongest categorical driver
(df.assign(flag=(df.Churn=='Yes'))
   .groupby('Contract')['flag'].mean().sort_values(ascending=False)
   .map('{:.1%}'.format))

Contract
Month-to-month    42.6%
One year          11.3%
Two year           2.8%
Name: flag, dtype: object

## 3. Modeling & evaluation
Three classifiers compared in a leakage-safe pipeline; best chosen by ROC-AUC with attention to recall.

In [6]:
from train_model import train_and_evaluate
metrics, best = train_and_evaluate()
metrics.round(3)

[split] train=5,616  test=1,405  churn(train)=26.4%
[fit]   Logistic Regression  AUC=0.840  Recall=0.774


[fit]   Random Forest        AUC=0.843  Recall=0.780


[fit]   Gradient Boosting    AUC=0.836  Recall=0.497

[metrics]
              Model  Accuracy  Precision  Recall    F1  ROC_AUC
      Random Forest     0.761      0.533   0.780 0.633    0.843
Logistic Regression     0.740      0.505   0.774 0.611    0.840
  Gradient Boosting     0.796      0.649   0.497 0.563    0.836

[best]  Random Forest
              precision    recall  f1-score   support

    Retained       0.90      0.75      0.82      1033
     Churned       0.53      0.78      0.63       372

    accuracy                           0.76      1405
   macro avg       0.72      0.77      0.73      1405
weighted avg       0.81      0.76      0.77      1405



[fig]  06_roc_curves.png


[fig]  07_confusion_matrix.png


[fig]  08_feature_importance.png


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
1,Random Forest,0.761,0.533,0.780,0.633,0.843
0,Logistic Regression,0.740,0.505,0.774,0.611,0.840
2,Gradient Boosting,0.796,0.649,0.497,0.563,0.836


## 4. Business impact
Revenue at risk, campaign ROI, and the ranked retention call-list.

In [7]:
import business_impact
rar, roi = business_impact.run()


  BUSINESS IMPACT (hold-out test set)
  Churners in test set      : 372
  Monthly revenue at risk   : $26,973
  Annualised revenue at risk: $323,680
  Avg monthly value/churner : $72.51
------------------------------------------------------------
  Strategy: target riskiest 20% (281 customers)
  Real churners caught      : 185 (precision 66%)
  Expected customers saved  : 56
  Campaign cost             : $26,976
  Revenue protected         : $53,717
  Net benefit               : $26,741
  ROI                       : 99%



[fig]  09_cumulative_gains.png
[csv]  retention_priority_list.csv  (top 250 at-risk customers)


In [8]:
# Peek at the actionable output: who to call first
pd.read_csv(config.MODELS_DIR / 'retention_priority_list.csv').head(10)

,Priority,tenure,Contract,MonthlyCharges,InternetService,PaymentMethod,ChurnProbability,ActualChurn
0,1,1,Month-to-month,100.80,Fiber optic,Electronic check,0.946756,1
1,2,1,Month-to-month,78.95,Fiber optic,Electronic check,0.938663,1
2,3,1,Month-to-month,79.60,Fiber optic,Electronic check,0.935503,1
3,4,1,Month-to-month,86.60,Fiber optic,Electronic check,0.935034,1
4,5,1,Month-to-month,81.10,Fiber optic,Electronic check,0.934411,1
5,6,1,Month-to-month,86.05,Fiber optic,Electronic check,0.934261,1
6,7,1,Month-to-month,93.30,Fiber optic,Electronic check,0.929295,1
7,8,1,Month-to-month,77.15,Fiber optic,Electronic check,0.928187,1
8,9,5,Month-to-month,100.50,Fiber optic,Electronic check,0.928057,1
9,10,6,Month-to-month,98.25,Fiber optic,Electronic check,0.925471,1


## 5. Takeaways
- Churn is **predictable** (ROC-AUC ~0.84) and **concentrated** in clear segments.
- The biggest lever is **contract type** (month-to-month vs annual).
- A model-guided campaign targeting the riskiest 20% delivers **~99% ROI**.

See `README.md` and `reports/EXECUTIVE_SUMMARY.md` for the full write-up.